# Analyse des performances et de la valeur marchande
Real Transfermarkt-derived historical study. Run the pipeline first; this notebook inspects committed outputs. All prices are EUR. No synthetic player data is used for the study.

In [ ]:
from pathlib import Path
import os, json
if Path.cwd().name == 'notebooks': os.chdir('..')
import pandas as pd
from IPython.display import display, Image
data = pd.read_csv('data/processed/player_seasons.csv')
metrics = json.loads(Path('reports/metrics.json').read_text())
display(data.head())
display(data.groupby('year').size().rename('player_seasons'))

## Qualité et sélection
Only players with at least 450 minutes and a recorded summer valuation are included. Snapshot positions may introduce look-ahead bias. Inspect the audit and date boundaries before interpreting results.

In [ ]:
display(pd.Series(json.loads(Path('reports/data_audit.json').read_text())['missing_percent'], name='missing_percent'))
assert (pd.to_datetime(data.last_match_date) < pd.to_datetime(data.cutoff)).all()
assert (pd.to_datetime(data.target_date) >= pd.to_datetime(data.cutoff)).all()
display(Image('reports/figures/correlations.png'))

## Modèles et validation temporelle
Hyperparameters use expanding folds through 2022; 2023 selects the estimator, 2024 calibrates intervals, 2025 is the final test. Compare persistence on the matched subset.

In [ ]:
display(pd.DataFrame(metrics['test_2025']).T)
display(Image('reports/figures/predictions.png'))
display(pd.read_csv('reports/importance.csv'))

## Profils automatiques
KMeans uses five performance indicators and excludes prices. PCA is only a projection; cluster labels are arbitrary.

In [ ]:
display(pd.read_csv('reports/cluster_summary.csv'))
display(Image('reports/figures/profiles.png'))

## Prix réels des transferts
Reported paid fees and strictly pre-transfer estimates are separate variables. Associations are not causal; free and undisclosed moves are excluded.

In [ ]:
display(pd.Series(json.loads(Path('reports/transfer_results.json').read_text())))
display(Image('reports/figures/transfer_fees.png'))

## Exemple de prédiction
Train locally to create trusted model artifacts: `python -m src.train`. Then run the following cell.

In [ ]:
from src.predict import predict
if Path('artifacts/valuation.joblib').exists():
    display(predict(pd.read_csv('examples/player.csv')))
else:
    print('Run python -m src.train to create the local models.')